In [1]:
# scraper/urls.py
def load_urls(filename="case_urls.txt"):
    with open(filename, "r") as file:
        return [line.strip() for line in file if line.strip()]


In [2]:
# scraper/fetch.py
import time
import random
import requests

session = requests.Session()

user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_4) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/81.0.4044.138 Safari/537.36",
    "Mozilla/5.0 (Windows NT 6.1; WOW64; rv:40.0) Gecko/20100101 Firefox/40.0"
]

def get_headers():
    return {"User-Agent": random.choice(user_agents)}

def fetch_ready_page(url, session=session, headers=None, max_retries=5, delay=5):
    headers = headers or get_headers()
    for attempt in range(max_retries):
        response = session.get(url, headers=headers)
        if response.status_code == 200:
            return response
        print(f"Attempt {attempt + 1}: Status {response.status_code}, retrying in {delay} seconds...")
        time.sleep(delay)
    print(f"Failed to fetch {url} after {max_retries} attempts.")
    return None


In [3]:
# scraper/parser.py
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime
import re

def parse_case_page(html, url):
    base_url = "https://www.courtlistener.com"
    soup = BeautifulSoup(html, "html.parser")

    full_title = soup.title.string.strip() if soup.title else "N/A"
    title = full_title.split(", 1")[0]

    entry_divs = soup.find_all("div", id=lambda x: x and x.startswith("entry-"))

    latest_date = "N/A"
    latest_link = "N/A"

    if entry_divs:
        latest_entry = entry_divs[-1]
        entry_text = latest_entry.get_text(separator=" ", strip=True)

        date_match = re.search(r"[A-Z][a-z]{2} \d{1,2}, \d{4}", entry_text)
        if date_match:
            raw_date = date_match.group(0)
            try:
                parsed_date = datetime.strptime(raw_date, "%b %d, %Y")
                latest_date = parsed_date.strftime("%Y-%m-%d")
            except ValueError:
                latest_date = raw_date

        a_tag = latest_entry.find("a", href=True)
        if a_tag:
            latest_link = urljoin(base_url, a_tag["href"])

    return {
        "Title": title,
        "Link": url,
        "Date of latest entry": latest_date,
        "Link of latest entry": latest_link
    }


In [4]:
# scraper/pipeline.py
import csv

def write_to_csv(rows, filename="output.csv"):
    if not rows:
        print("No data to write.")
        return
    with open(filename, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved {len(rows)} rows to {filename}")


In [5]:
# main.py
from scraper.urls import load_urls
from scraper.fetch import fetch_ready_page
from scraper.parser import parse_case_page
from scraper.pipeline import write_to_csv

def main():
    urls = load_urls()
    rows = []

    for url in urls:
        response = fetch_ready_page(url, delay=2)
        if response is None:
            print(f"Skipping {url}")
            continue

        row = parse_case_page(response.text, url)
        rows.append(row)

    write_to_csv(rows)

if __name__ == "__main__":
    main()


FileNotFoundError: [Errno 2] No such file or directory: 'case_urls.txt'